# Credit scenarios

Spread widening via **par CDS / hazard** bumps (uniform and id-specific), **`instrument_price_pct_by_attr`** for sector-style shocks on tagged instruments, and a short note on **rating migration** as a conceptual step beyond parallel spreads.

## Concept

- **Uniform widening**: `OperationSpec.curve_parallel_bp` with `CurveKind.par_cds()` shifts quoted spreads, which the engine maps into **hazard** space (with a discount curve anchor).
- **Sector / name differentiation**: bump one hazard id more than another, or target **`OperationSpec.instrument_price_pct_by_attr`** when instruments carry metadata (e.g. `sector=Energy`).
- **Rating migration** (conceptual): a notch downgrade implies a **new spread curve** or **higher hazard**, often larger than a parallel +X bp stress—typically modeled as a **regime change**, not a tiny bump.

We compare **survival** \(S(t)\) at selected horizons before and after shocks.

In [1]:
import json
from pathlib import Path

from finstack_quant.scenarios import (
    CurveKind,
    OperationSpec,
    apply_scenario_to_market,
    build_from_template,
    build_scenario_spec,
    build_template_component,
    compose_scenarios,
    list_builtin_templates,
    validate_scenario_spec,
)

print(
    "Imports OK",
    callable(compose_scenarios),
    callable(apply_scenario_to_market),
    callable(build_template_component),
)

print("Templates available:", list_builtin_templates())

tmeta = build_from_template("covid_2020")
print("Sample historical template covid_2020 (credit excerpt search):")
print(tmeta[:400], "...")


def ops_json(*ops: OperationSpec) -> str:
    """Serialize typed operations into the JSON list `build_scenario_spec` expects."""
    return json.dumps([json.loads(op.to_json()) for op in ops])


PAR_CDS = CurveKind.par_cds()

# Wire format (the one raw-JSON example in this notebook): operations are plain JSON
# objects tagged by `kind`, so a scenario can live in a config file and be handed
# straight to `build_scenario_spec`. Everything below uses the typed builders.
ops_uniform = json.loads(Path("data/credit_scenarios.json").read_text())["ops_uniform"]
su = build_scenario_spec("credit_uniform", json.dumps(ops_uniform), "Uniform +75bp", "All buckets widen")
print("validate uniform:", validate_scenario_spec(su))

# Sector-style: widen HY more than IG (two curve IDs). `par_cds` shocks need a rate
# anchor, supplied as the trailing `discount_curve_id` argument.
ss = build_scenario_spec(
    "credit_sector",
    ops_json(
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 25.0, "USD-OIS"),
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HY", 150.0, "USD-OIS"),
    ),
    "IG vs HY",
    "HY wider",
)
print("validate sector:", validate_scenario_spec(ss))

# Metadata-based price shock. The builder takes an ORDERED list of `(key, value)`
# pairs; hand-written JSON that spells `attrs` as a bare map is exactly the shape
# drift the typed path removes.
sa = build_scenario_spec(
    "energy_px",
    ops_json(OperationSpec.instrument_price_pct_by_attr([("sector", "Energy")], -2.5)),
    "Energy price",
    "-2.5% on Energy-tagged instruments",
)
print("validate attr shock:", validate_scenario_spec(sa))
print("Built three credit-related scenario specs (uniform, id-specific, attribute).")


Imports OK True True True
Templates available: ['gfc_2008', 'covid_2020', 'rate_shock_2022', 'svb_2023', 'ltcm_1998']
Sample historical template covid_2020 (credit excerpt search):
{"id":"covid_2020","name":"COVID 2020 Market Shock","description":"Pandemic panic - emergency easing, spread widening, equity crash, vol spike","operations":[{"kind":"curve_parallel_bp","curve_kind":"discount","curve_id":"USD-SOFR","bp":-150.0},{"kind":"curve_node_bp","curve_kind":"discount","curve_id":"USD-SOFR","nodes":[["2Y",-30.0],["10Y",10.0],["30Y",20.0]],"match_mode":"interpolate"},{"kind": ...
validate uniform: True
validate sector: True
validate attr shock: True
Built three credit-related scenario specs (uniform, id-specific, attribute).


In [2]:
import sys
sys.path.insert(0, "..")

from _shared import DEMO_AS_OF
from finstack_quant.core.market_data import DiscountCurve, HazardCurve, MarketContext
from finstack_quant.valuations.instruments import price_instrument

bd = DEMO_AS_OF
AS_OF = str(bd)

# Bespoke IG/HY pair rather than `_shared.build_demo_market`: that market carries a
# single `CORP-HAZARD` curve, and the differentiated-stress lesson below needs two
# credit curves at different quality tiers.
usd = DiscountCurve("USD-OIS", bd, [(0.0, 1.0), (1.0, 0.98), (5.0, 0.90)], day_count="act_365f")
ig = HazardCurve(
    "CORP-IG",
    bd,
    [(1.0, 0.012), (3.0, 0.018), (5.0, 0.022)],
    recovery_rate=0.4,
    par_spreads=[(1.0, 72.189173), (3.0, 96.787825), (5.0, 110.547701)],
)
hy = HazardCurve(
    "CORP-HY",
    bd,
    [(1.0, 0.045), (3.0, 0.055), (5.0, 0.06)],
    recovery_rate=0.35,
    par_spreads=[(1.0, 293.404024), (3.0, 336.773666), (5.0, 356.253819)],
)

mc = MarketContext().insert(usd).insert(ig).insert(hy)
base_json = mc.to_json()

base_ig = mc.get_hazard("CORP-IG")
base_hy = mc.get_hazard("CORP-HY")
for t in (1.0, 3.0, 5.0):
    print(f"Base survival CORP-IG t={t}Y S={base_ig.sp(t):.6f}  HY S={base_hy.sp(t):.6f}")

# Cross-check: a 0bp par-CDS bump must round-trip the curve through the engine's
# spread -> hazard calibration and land back on the original survival probabilities.
zero_spec = build_scenario_spec(
    "zero_credit_bump",
    ops_json(
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 0.0, "USD-OIS"),
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HY", 0.0, "USD-OIS"),
    ),
    "Zero-bump calibration check",
    "",
)
zero_market_json = apply_scenario_to_market(zero_spec, base_json, AS_OF)["market_json"]
zero_market = MarketContext.from_json(zero_market_json)
zero_ig = zero_market.get_hazard("CORP-IG")
zero_hy = zero_market.get_hazard("CORP-HY")
for t in (1.0, 3.0, 5.0):
    assert abs(base_ig.sp(t) - zero_ig.sp(t)) < 1e-8
    assert abs(base_hy.sp(t) - zero_hy.sp(t)) < 1e-8

def apply_and_print(label: str, spec_json: str):
    result = apply_scenario_to_market(spec_json, base_json, AS_OF)
    print(f"\n--- {label} ---")
    print(
        "  operations_applied:",
        result["operations_applied"],
        " warnings:",
        result["warnings"],
    )
    stressed_market = MarketContext.from_json(result["market_json"])
    stressed_ig = stressed_market.get_hazard("CORP-IG")
    stressed_hy = stressed_market.get_hazard("CORP-HY")
    for t in (1.0, 3.0, 5.0):
        print(
            f"  t={t}Y  IG S {base_ig.sp(t):.6f} -> {stressed_ig.sp(t):.6f}  |  HY {base_hy.sp(t):.6f} -> {stressed_hy.sp(t):.6f}"
        )

spec_uniform = build_scenario_spec(
    "u",
    ops_json(OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 80.0, "USD-OIS")),
    "IG +80bp",
    "Uniform widening on the IG curve only",
)

spec_both = build_scenario_spec(
    "b",
    ops_json(
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-IG", 30.0, "USD-OIS"),
        OperationSpec.curve_parallel_bp(PAR_CDS, "CORP-HY", 120.0, "USD-OIS"),
    ),
    "IG +30bp, HY +120bp",
    "Differentiated widening across quality tiers",
)

apply_and_print("IG-only +80bp par CDS shock", spec_uniform)
apply_and_print("Sector-style IG +30bp, HY +120bp", spec_both)

# Reprice a single-name CDS off CORP-IG under base vs IG-only stressed market.
# Written out here rather than taken from `_shared.instrument_fixtures.cds` because
# that fixture references `CORP-HAZARD`; this trade must reference `CORP-IG`.
cds_ig = json.dumps(
    {
        "type": "credit_default_swap",
        "spec": {
            "id": "CDS-ON-IG",
            "notional": {"amount": 1_000_000.0, "currency": "USD"},
            "side": "pay",
            "convention": "isda_na",
            "premium": {
                "start": AS_OF,
                "end": "2030-01-15",
                "frequency": {"count": 3, "unit": "months"},
                "stub": "ShortFront",
                "bdc": "following",
                "calendar_id": None,
                "day_count": "Act360",
                "spread_bp": 120.0,
                "discount_curve_id": "USD-OIS",
            },
            "protection": {
                "credit_curve_id": "CORP-IG",
                "recovery_rate": 0.4,
                "settlement_delay": 3,
            },
            "pricing_overrides": {},
            "attributes": {},
        },
    }
)

def cds_pv_usd(mj: str) -> float:
    rj = price_instrument(cds_ig, mj, AS_OF, model="hazard_rate")
    return float(json.loads(rj)["value"]["amount"])

ru = apply_scenario_to_market(spec_uniform, base_json, AS_OF)
pv_base = cds_pv_usd(base_json)
pv_stress = cds_pv_usd(ru["market_json"])
print(
    f"\nCDS (CORP-IG) PV — base vs IG-only +80bp market: {pv_base:,.2f} -> {pv_stress:,.2f} (Δ {pv_stress - pv_base:+,.2f} USD)"
)

print("\nRating migration (conceptual): a multi-notch move usually implies re-calibrating to a new curve level/shape, not a single parallel bump.")


Base survival CORP-IG t=1.0Y S=0.988072  HY S=0.955997
Base survival CORP-IG t=3.0Y S=0.953134  HY S=0.856415
Base survival CORP-IG t=5.0Y S=0.912105  HY S=0.759572



--- IG-only +80bp par CDS shock ---
  operations_applied: 1  warnings: []
  t=1.0Y  IG S 0.988072 -> 0.975023  |  HY 0.955997 -> 0.955997
  t=3.0Y  IG S 0.953134 -> 0.915771  |  HY 0.856415 -> 0.856415
  t=5.0Y  IG S 0.912105 -> 0.853120  |  HY 0.759572 -> 0.759572



--- Sector-style IG +30bp, HY +120bp ---
  operations_applied: 2  warnings: []
  t=1.0Y  IG S 0.988072 -> 0.983158  |  HY 0.955997 -> 0.938579
  t=3.0Y  IG S 0.953134 -> 0.938947  |  HY 0.856415 -> 0.810229
  t=5.0Y  IG S 0.912105 -> 0.889523  |  HY 0.759572 -> 0.692250



CDS (CORP-IG) PV — base vs IG-only +80bp market: -5,556.16 -> 31,128.02 (Δ +36,684.18 USD)

Rating migration (conceptual): a multi-notch move usually implies re-calibrating to a new curve level/shape, not a single parallel bump.


## Takeaways

- **`CurveKind.par_cds()`** shocks are the primary way to stress **credit curves** tied to **`HazardCurve`** ids; pass **`discount_curve_id`** so the engine has a rate anchor for the spread → hazard calibration.
- **Uniform vs differentiated** credit stress is expressed by **one shared bump** or **different `curve_id` targets** (and magnitudes) in the same operation list.
- **`OperationSpec.instrument_price_pct_by_attr`** is useful when positions carry **metadata** you want to shock as a group; it takes an **ordered list of `(key, value)` pairs**, which is why the typed builder is safer than hand-writing the `attrs` object.
- A **0bp bump** is a cheap calibration cross-check: it round-trips the curve through spread → hazard and must return the original survival probabilities.
- **Survival** drops when spreads widen—deserialize stressed **`market_json`** with **`MarketContext.from_json`** and use **`get_hazard`** to inspect \(S(t)\).